In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import numpy as np
import napari
import zarr
import dask.array as da

from scipy import ndimage as ndi
from skimage import io
from tqdm import tqdm
import pandas as pd
from vispy.color import get_colormap

from morphotrack import analysis, utils
import open3d as o3d

from ome_zarr.io import parse_url
from ome_zarr.reader import Reader


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def color_map_order(size, color_code='viridis'):
    """
    return colors for point visualization.
    """
    code = get_colormap(color_code).map(
        np.arange(size) / size
    )

    return code

In [3]:
# load tracks
tracks_asma = pd.read_csv("/mnt/ampa_data01/tmurakami/220806_visual_02_R01/morphotrack/R01_ch561_10um.tif_traces.csv")

scale_micron = np.asarray((10,10,10))

roi_num_col = "ROI_Number"
coord_cols = ['Z_coord','Y_coord','X_coord']

img_asma = io.imread("/mnt/ampa_data01/tmurakami/220806_visual_02_R01/morphotrack/R01_ch561_10um.tif")

In [4]:
# confirm the order of the points is from out to in.
viewer = napari.Viewer()

tr = tracks_asma
colors = []
track_points = []
for i in tr[roi_num_col].unique():
    track = tr[tr[roi_num_col] == i]
    track_p = track[coord_cols].to_numpy()
    colors.append(color_map_order(track_p.shape[0]))
    track_points.append(track_p)
colors = np.vstack(colors)

points_asma = tracks_asma[coord_cols].to_numpy() * scale_micron

track_points = np.vstack(track_points)
viewer.add_points(track_points,edge_width=0,face_color=colors,blending='translucent_no_depth',scale=scale_micron)
viewer.add_image(img_asma,blending='additive',colormap='gray',scale=scale_micron)

/tmp/ipykernel_3746890/840989547.py:17: FutureWarning: Argument 'edge_width' is deprecated, please use 'border_width' instead. The argument 'edge_width' was deprecated in 0.5.0 and it will be removed in 0.6.0.
  viewer.add_points(track_points,edge_width=0,face_color=colors,blending='translucent_no_depth',scale=scale_micron)


<Image layer 'img_asma' at 0x7f42b8731e10>

In [5]:
local_vectors = []
track_positions = []

get_all_pos = True
if get_all_pos:
    tr = tracks_asma
    points = points_asma
    for i in tr[roi_num_col].unique():
        extract = tr[roi_num_col] == i
        track = tr[extract]
        track_position = points[extract]
        local_vecs = np.diff(track_position,n=1,axis=0)
        local_vectors.append(local_vecs)
        track_position = track_position[1:,:]
        track_positions.append(track_position)
    local_vectors = np.vstack(local_vectors)
    track_positions = np.vstack(track_positions)
else:
    tr = tracks_asma
    points = points_asma
    for i in tr[roi_num_col].unique():
        extract = tr[roi_num_col] == i
        track = tr[extract]
        track_position = points[extract]
        local_vecs = np.diff(track_position,n=1,axis=0)
        local_vecs = local_vecs[[0,-1],:]
        local_vectors.append(local_vecs)
        track_position = track_position[[0,-2],:]
        track_positions.append(track_position)
    local_vectors = np.vstack(local_vectors)
    track_positions = np.vstack(track_positions)

In [17]:
viewer = napari.Viewer()
viewer.add_vectors(np.stack([track_positions, local_vectors], axis=1), length=10, edge_width=20, edge_color='red', name='3D Vectors',out_of_slice_display=True)
viewer.add_image(img_asma,blending='additive',colormap='gray',scale=scale_micron)

<Image layer 'img_asma' at 0x7f2cc42a2dd0>

In [6]:
if get_all_pos:
    np.save("/home/tmurakami/src/flow_analysis/human_analysis/02_output/visual_02_R01/positions_sma_dense.npy",track_positions)
    np.save("/home/tmurakami/src/flow_analysis/human_analysis/02_output/visual_02_R01/vectors_sma_dense.npy",local_vectors)
else:
    np.save("/home/tmurakami/src/flow_analysis/human_analysis/02_output/visual_02_R01/positions_sma_sparse.npy",track_positions)
    np.save("/home/tmurakami/src/flow_analysis/human_analysis/02_output/visual_02_R01/vectors_sma_sparse.npy",local_vectors)